# Week 6 - Pandas Lab\nAll steps from manual

In [ ]:
import pandas as pd, numpy as np\nprint(pd.__version__)

In [ ]:
students=pd.read_csv('week06/data/raw/students_messy.csv', na_values=['','missing','N/A','-'])\nprint(students.head())\nprint(students.shape)\nprint(students.isna().sum())\nprint('dup',students.duplicated().sum())

In [ ]:
students=students.drop_duplicates().reset_index(drop=True)\n# text clean\nstudents['program']=students['program'].astype('string').str.strip().str.upper()\nstudents['program']=students['program'].replace({'BS AI':'BSAI','BS-AI':'BSAI','AI':'BSAI','BS DS':'BSDS'})\nstudents['city']=students['city'].astype('string').str.strip().str.title()\nstudents['city']=students['city'].replace({'Rwp':'Rawalpindi'})\nstudents['city']=students['city'].fillna('Unknown')\nprint(students[['program','city']].head())

In [ ]:
# types\nfor c in ['attendance','assignment','midterm','final']:\n students[c]=pd.to_numeric(students[c], errors='coerce')\nstudents['admission_date']=pd.to_datetime(students['admission_date'], dayfirst=True, errors='coerce')\nstudents['student_id']=students['student_id'].astype('string')\n# range\ncleaning_log=[]; invalid=[]\nfor c in ['attendance','assignment','midterm','final']:\n m=(students[c]<0)|(students[c]>100)\n if m.any():\n cleaning_log.append({'col':c,'action':'set NA'})\n students.loc[m,c]=pd.NA\n# impute\nfor c in ['attendance']:\n students[c]=students.groupby('program')[c].transform(lambda x: x.fillna(x.median()))\n students[c]=students[c].fillna(students[c].median())\nfor c in ['assignment','midterm','final']:\n students[c]=students[c].fillna(students[c].median())\nprint(students.isna().sum())

In [ ]:
# features\nstudents['weighted_score']=students['assignment']*0.2+students['midterm']*0.3+students['final']*0.5\nimport numpy as np\nstudents['attendance_status']=np.select([students['attendance']<75,(students['attendance']>=75)&(students['attendance']<85),students['attendance']>=85],['Low','Satisfactory','Excellent'])\nstudents['grade']=np.select([students['weighted_score']>=85,students['weighted_score']>=75,students['weighted_score']>=65,students['weighted_score']>=50],['A','B','C','D'],default='F')\nstudents['at_risk']=(students['attendance']<75)|(students['weighted_score']<50)\nprogram_summary=students.groupby('program').agg(student_count=('student_id','count'),avg_weighted_score=('weighted_score','mean')).reset_index()\ngrade_distribution=students.groupby('grade').agg(count=('student_id','count')).reset_index()\nprint(program_summary)\nprint(grade_distribution)\n# save\nstudents.to_csv('week06/data/processed/students_clean.csv',index=False)\nprogram_summary.to_csv('week06/data/processed/program_summary.csv',index=False)\ngrade_distribution.to_csv('week06/data/processed/grade_distribution.csv',index=False)\nprint('saved')